# Assignment 3: Image Classification using CNN (MNIST)
**Aim:** Build an image classification model using a CNN in 4 stages:

a. Loading and preprocessing the image data  b. Defining the model architecture  c. Training the model  d. Estimating performance

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

I0000 00:00:1793640566.577904   36870 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793640568.338351   36870 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## a. Loading and preprocessing the image data

In [2]:
# Path of dataset folder (change if your folder is somewhere else)
train_dir = "dataset/mnist-jpg/train"
test_dir  = "dataset/mnist-jpg/test"

datagen = ImageDataGenerator(rescale=1.0/255)   # pixel values 0-255 -> 0-1

train_gen = datagen.flow_from_directory(train_dir, target_size=(28, 28),
                                        color_mode="grayscale", batch_size=60000,
                                        class_mode="categorical", shuffle=True)
test_gen  = datagen.flow_from_directory(test_dir, target_size=(28, 28),
                                        color_mode="grayscale", batch_size=10000,
                                        class_mode="categorical", shuffle=True)

# One big batch = whole dataset loaded as numpy arrays
x_train, y_train = train_gen[0]
x_test,  y_test  = test_gen[0]
class_names = list(train_gen.class_indices.keys())

print("x_train:", x_train.shape, " y_train:", y_train.shape)
print("x_test :", x_test.shape,  " y_test :", y_test.shape)
print("Classes:", class_names)

Found 60000 images belonging to 10 classes.


Found 10000 images belonging to 10 classes.


x_train: (60000, 28, 28, 1)  y_train: (60000, 10)
x_test : (10000, 28, 28, 1)  y_test : (10000, 10)
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [3]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[np.argmax(y_train[i])])
    plt.axis("off")
plt.show()

/tmp/ipykernel_36870/273738373.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## b. Defining the model's architecture
Conv2D (feature extraction with filters) → MaxPooling (downsampling) → Flatten → Dense → Softmax output (10 classes)

In [4]:
model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

## c. Training the model

In [5]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(x_train, y_train,
                    validation_data=(x_test, y_test),
                    epochs=5, batch_size=64)

Epoch 1/5


  1/938 ━━━━━━━━━━━━━━━━━━━━ 8:34 549ms/step - accuracy: 0.1719 - loss: 2.3042

  5/938 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.2562 - loss: 2.2316  

 10/938 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.3516 - loss: 2.1050

 15/938 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.4062 - loss: 1.9388

 20/938 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.4602 - loss: 1.7647

... (967 outputs left out)

925/938 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9902 - loss: 0.0310

930/938 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9902 - loss: 0.0310

935/938 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9903 - loss: 0.0309

938/938 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.9902 - loss: 0.0309 - val_accuracy: 0.9910 - val_loss: 0.0260


## d. Estimating the model's performance

In [6]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d",
            xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

Test loss: 0.0260
Test accuracy: 99.10%


              precision    recall  f1-score   support

           0       0.99      0.99      0.99       980
           1       0.99      1.00      1.00      1135
           2       0.99      0.99      0.99      1032
           3       0.99      1.00      0.99      1010
           4       1.00      0.98      0.99       982
           5       0.99      0.99      0.99       892
           6       0.99      0.99      0.99       958
           7       0.99      0.99      0.99      1028
           8       0.99      0.99      0.99       974
           9       0.98      0.99      0.98      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000



/tmp/ipykernel_36870/4274663811.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.show()

/tmp/ipykernel_36870/1834603888.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
n = 25   # change index to test another image
plt.imshow(x_test[n].squeeze(), cmap="gray"); plt.axis("off"); plt.show()
pred = model.predict(x_test[n:n+1], verbose=0)
print("Predicted:", class_names[np.argmax(pred)])
print("Actual   :", class_names[np.argmax(y_test[n])])

Predicted: 9
Actual   : 9


/tmp/ipykernel_36870/3822332667.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(x_test[n].squeeze(), cmap="gray"); plt.axis("off"); plt.show()
